In [9]:
%run simplexalgo.ipynb


M/m m
number of variables 1
c_0 =  1
number of constraints 1
a_0,0 =  1
b_0 =  1


min z = 1.0x1

Subject to

1.0x1 <= 1.0
optimal solution is [1.]
optimal value is 1.0


In [2]:
# =========================
# DATA GENERATION 
# =========================

MAX_VARS = 7
MAX_CONS = 7

def generate_lpp(n_vars, n_constraints):
    c = np.random.uniform(-3, 7, size=(n_vars,))
    A = np.random.uniform(-5, 9, size=(n_constraints, n_vars))
    b = np.random.uniform(1, 30, size=(n_constraints,))  # avoid zeros
    o = random.choice(['M','m'])

    # padding
    c = np.pad(c, (0, MAX_VARS - n_vars))
    A = np.pad(A, ((0, MAX_CONS - n_constraints), (0, MAX_VARS - n_vars)))
    b = np.pad(b, (0, MAX_CONS - n_constraints))

    return o, c, A, b


def generate_dataset(n_samples, n_vars, n_constraints):
    data = []
    for _ in range(n_samples):
        o, c, A, b = generate_lpp(n_vars, n_constraints)
        X = simplex(o, c.copy(), A.copy(), b.copy())

        if X is None:
            continue
        if np.any(np.abs(X) > 20):   # tighter filter
            continue

        data.append((o, c, A, b, X))

    return data


# =========================
# BUILD DATASET
# =========================

dataset = []
for n in range(3, MAX_VARS):
    for m in range(1, MAX_CONS):
        dataset += generate_dataset(10000, n, m)

print("Dataset size:", len(dataset))


# =========================
# NORMALIZATION
# =========================

def normalize_instance(A, b, c):
    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    c = c / (np.linalg.norm(c) + 1e-8)
    return A, b, c


Dataset size: 178259


In [3]:
# =========================
# DATASET CLASS
# =========================

train_data, test_data = train_test_split(dataset, test_size=0.15, random_state=42)

class LPPDataset(Dataset):
    def __init__(self, data):
        self.X, self.flags, self.c, self.y = [], [], [], []

        for (o, c, A, b, X) in data:
            A, b, c = normalize_instance(A, b, c)

            mat = np.concatenate([A, b.reshape(-1,1)], axis=1)
            flag = np.array([1.0 if o == 'M' else -1.0])

            self.X.append(mat)
            self.flags.append(flag)
            self.c.append(c)
            self.y.append(X)

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.flags = torch.tensor(np.array(self.flags), dtype=torch.float32)
        self.c = torch.tensor(np.array(self.c), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.flags[idx], self.c[idx], self.y[idx]


train_data, test_data = train_test_split(dataset, test_size=0.15, random_state=42)

train_loader = DataLoader(LPPDataset(train_data), batch_size=64, shuffle=True)
test_loader  = DataLoader(LPPDataset(test_data),  batch_size=64)

In [4]:
# =========================
# MODEL
# =========================

class Model(nn.Module):
    def __init__(
        self,
        in_dim=8,            # MAX_VARS + 1
        phi_hidden=128,
        mlp_hidden1=256,
        mlp_hidden2=128,
        mlp_hidden3=64,
        c_dim=7,            # MAX_VARS
        output_dim=7
    ):
        super().__init__()

        self.phi = nn.Sequential(
            nn.Linear(in_dim, phi_hidden),
            nn.LayerNorm(phi_hidden),
            nn.ReLU(),
            nn.Linear(phi_hidden, phi_hidden),
            nn.LayerNorm(phi_hidden),
            nn.ReLU(),
        )

        self.fc1 = nn.Linear(phi_hidden*2 + 1 + c_dim, mlp_hidden1)
        self.fc2 = nn.Linear(mlp_hidden1, mlp_hidden2)
        self.fc3 = nn.Linear(mlp_hidden2, mlp_hidden3)
        self.fc4 = nn.Linear(mlp_hidden3, output_dim)

        self.act = nn.ReLU()

    def forward(self, x, o, c):

        x = self.phi(x)

        x_mean = x.mean(dim=1)
        x_max  = x.max(dim=1).values

        x = torch.cat([x_mean, x_max, o, c], dim=1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        x = self.fc4(x)

        return F.softplus(x)   # keeps gradients alive


In [5]:
def total_loss(X, pred, c, o, y):

    # regression
    mse = ((pred - y)**2).mean()

    # constraints
    A = X[:, :, :-1]
    b = X[:, :, -1]
    Ax = torch.bmm(A, pred.unsqueeze(-1)).squeeze(-1)

    constr = (torch.relu(Ax - b)**2).mean()
    nonneg = (torch.relu(-pred)**2).mean()

    return mse + 7*(constr + nonneg)

In [6]:
# =========================
# TRAINING
# =========================

model = Model()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 60

for epoch in range(EPOCHS):

    model.train()
    total = 0

    for X, o, c, y in train_loader:

        pred = model(X, o, c)

        loss = total_loss(X, pred, c, o, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(train_loader):.16f}")

Epoch 1: 3.0563211242693500
Epoch 2: 2.8895023055878042
Epoch 3: 2.8060575572119371
Epoch 4: 2.7075152639047921
Epoch 5: 2.5893415231761097
Epoch 6: 2.4803434310078218
Epoch 7: 2.3984709481985584
Epoch 8: 2.3371594991484606
Epoch 9: 2.2859355028013923
Epoch 10: 2.2410154287023722
Epoch 11: 2.1997787162309161
Epoch 12: 2.1571880263100200
Epoch 13: 2.1188751721150569
Epoch 14: 2.0870124497576743
Epoch 15: 2.0560712318968131
Epoch 16: 2.0332441820284806
Epoch 17: 2.0136932171629489
Epoch 18: 1.9892754971578315
Epoch 19: 1.9717343858871106
Epoch 20: 1.9524362690595758
Epoch 21: 1.9335227853593391
Epoch 22: 1.9140094213087011
Epoch 23: 1.8959396956740200
Epoch 24: 1.8767149492947233
Epoch 25: 1.8594135740469839
Epoch 26: 1.8415856022085693
Epoch 27: 1.8269315078804218
Epoch 28: 1.8112854286273186
Epoch 29: 1.7975224181002862
Epoch 30: 1.7854154986404889
Epoch 31: 1.7720296161400306
Epoch 32: 1.7626569544191699
Epoch 33: 1.7514333829686448
Epoch 34: 1.7414165425391213
Epoch 35: 1.72923279906

In [11]:
#accuracy

total_violation = 0
total_obj_gap = 0
total_samples = 0

with torch.no_grad():
    for X_batch, flag_batch, c_batch, y_batch in test_loader:

        pred = model(X_batch, flag_batch, c_batch)

        batch_size = X_batch.size(0)


        # constraint violation
        A = X_batch[:, :, :-1]
        b = X_batch[:, :, -1]
        Ax = torch.bmm(A, pred.unsqueeze(-1)).squeeze(-1)
        violation = torch.relu(Ax - b).mean()

        #  objective gap
        true_obj = (c_batch * y_batch).sum(dim=1)
        pred_obj = (c_batch * pred).sum(dim=1)
        obj_gap = (torch.abs(true_obj - pred_obj) /
           (torch.abs(true_obj) + 1.0)).mean()


        total_violation += violation.item() * batch_size
        total_obj_gap   += obj_gap.item() * batch_size
        total_samples   += batch_size

n = total_samples

print("========== TEST RESULTS ==========")
print(f"Constraint Violation:{total_violation/n:.4f}  (0 = perfect)")
print(f"Objective Gap:       {total_obj_gap/n:.4f} ")
print("==================================")

========== TEST RESULTS ==========
Constraint Violation:0.0573  (0 = perfect)
Objective Gap:       0.2743 
